# Recipe Scraper Utility

This notebook scrapes recipes from a given URL and saves them in the structured format required by the recipes website, downloading the main recipe image and updating the main index file automatically.

In [2]:
# Install dependencies if they are not already installed
!pip install recipe-scrapers beautifulsoup4 requests

  Obtaining dependency information for recipe-scrapers from https://files.pythonhosted.org/packages/75/5a/ff6ba9552d7551c719812d2e9d9847c57abee4ffa11941d84f87b84a8764/recipe_scrapers-15.11.0-py3-none-any.whl.metadata
  Obtaining dependency information for extruct>=0.17.0 from https://files.pythonhosted.org/packages/33/00/60d3efc09d52a9e8d57b59323d7edb962daba2f1c46b89449a252f3652e7/extruct-0.18.0-py2.py3-none-any.whl.metadata
  Obtaining dependency information for isodate>=0.6.1 from https://files.pythonhosted.org/packages/15/aa/0aca39a37d3c7eb941ba736ede56d689e7be91cab5d9ca846bde3999eba6/isodate-0.7.2-py3-none-any.whl.metadata
  Obtaining dependency information for lxml from https://files.pythonhosted.org/packages/62/b0/83f481780d1548750b8ce2ec824073deef2f452d9cd1a6faff8507e3d16d/lxml-6.1.1-cp311-cp311-macosx_10_9_universal2.whl.metadata
  Obtaining dependency information for lxml-html-clean from https://files.pythonhosted.org/packages/6a/bd/6e2b76a6c5dee10397db9c929f0c5066766ec1036046

In [3]:
import os
import re
import json
import requests
import mimetypes
from recipe_scrapers import scrape_me

In [4]:
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
}

UNITS_LIST = [
    "g", "kg", "grams", "gram", "kilograms", "kilogram", "oz", "ounce", "ounces", "lb", "lbs", "pound", "pounds",
    "ml", "l", "cl", "dl", "litre", "litres", "liter", "liters", "cup", "cups", "tbsp", "tbs", "tablespoon", "tablespoons",
    "tsp", "teaspoon", "teaspoons", "fl oz", "fluid ounces", "fluid ounce", "pint", "pints", "quart", "quarts", "gallon", "gallons",
    "can", "cans", "tin", "tins", "pack", "packs", "package", "packages", "bag", "bags", "box", "boxes", "bunch", "bunches",
    "clove", "cloves", "head", "heads", "sprig", "sprigs", "stalk", "stalks", "slice", "slices", "pinch", "pinches", "dash", "dashes",
    "drop", "drops", "handful", "handfuls", "bulb", "bulbs", "rasher", "rashers", "piece", "pieces", "cube", "cubes"
]
# Sort units by length descending to match multi-word/longer units first
UNITS_LIST.sort(key=len, reverse=True)

In [6]:
def parse_ingredient(raw_str):
    raw_str = raw_str.strip()
    
    # 1. Handle special starting case: "pinch of ..."
    if raw_str.lower().startswith("pinch of "):
        ingredient = raw_str[len("pinch of "):].strip()
        note = "pinch"
        paren_match = re.search(r'\(([^)]+)\)\s*$', ingredient)
        if paren_match:
            note = f"pinch, {paren_match.group(1).strip()}"
            ingredient = ingredient[:paren_match.start()].strip()
        if ',' in ingredient:
            parts = ingredient.split(',', 1)
            ingredient = parts[0].strip()
            note = f"pinch, {parts[1].strip()}"
        return {
            "amount": "",
            "unit": "",
            "ingredient": ingredient,
            "note": note
        }
        
    # 2. Extract note from ending parentheses or commas
    note = ""
    paren_match = re.search(r'\(([^)]+)\)\s*$', raw_str)
    if paren_match:
        note = paren_match.group(1).strip()
        raw_str = raw_str[:paren_match.start()].strip()
        
    if ',' in raw_str:
        parts = raw_str.split(',', 1)
        main_part = parts[0].strip()
        comma_note = parts[1].strip()
        if note:
            note = f"{comma_note} ({note})"
        else:
            note = comma_note
    else:
        main_part = raw_str
        
    # 3. Match leading amount (numbers, decimals, fractions, ranges)
    amount_regex = r'^((?:[0-9]+(?:\s*/\s*[0-9]+)?|[0-9]*[½⅓¼⅛⅔¾⅜⅝⅞]|[0-9]+\.[0-9]+|\.[0-9]+)(?:\s*(?:-|–|—|to)\s*(?:[0-9]+(?:\s*/\s*[0-9]+)?|[0-9]*[½⅓¼⅛⅔¾⅜⅝⅞]|[0-9]+\.[0-9]+|\.[0-9]+))?)'
    amount = ""
    rest = main_part
    
    amount_match = re.match(amount_regex, main_part)
    if amount_match:
        amount = amount_match.group(1).strip()
        rest = main_part[amount_match.end():].strip()
        
    # 4. Extract unit from the start of rest if it is in the units list
    unit = ""
    ingredient = rest
    
    for u in UNITS_LIST:
        pattern = r'^(' + re.escape(u) + r'\b\.?)(\s+(?:of\s+)?)?'
        unit_match = re.match(pattern, rest, re.IGNORECASE)
        if unit_match:
            unit = unit_match.group(1).lower().rstrip('.')
            ingredient = rest[unit_match.end():].strip()
            break
            
    # Clean up double spaces
    ingredient = re.sub(r'\s+', ' ', ingredient).strip()
    
    return {
        "amount": amount,
        "unit": unit,
        "ingredient": ingredient,
        "note": note
    }

In [7]:
def scrape_recipe(url):
    print(f"Scraping URL: {url} ...")
    
    try:
        scraper = scrape_me(url)
    except Exception as e:
        print(f"Error: Failed to scrape URL using recipe-scrapers: {e}")
        return
        
    title = scraper.title()
    print(f"Scraped Title: {title}")
    
    # 1. Determine next ID from public/recipes.json
    recipes_json_path = os.path.join("public", "recipes.json")
    recipes = []
    if os.path.exists(recipes_json_path):
        try:
            with open(recipes_json_path, "r", encoding="utf-8") as f:
                recipes = json.load(f)
        except Exception as e:
            print(f"Warning: Failed to load public/recipes.json: {e}")
            recipes = []
            
    if recipes:
        new_id = max(item.get("id", 0) for item in recipes) + 1
    else:
        new_id = 1
        
    print(f"Determined new recipe ID: {new_id}")
    
    # 2. Download and save the image
    image_url = scraper.image()
    saved_image_path = ""
    if image_url:
        print(f"Downloading image from: {image_url} ...")
        try:
            r = requests.get(image_url, headers=HEADERS, stream=True, timeout=15)
            if r.status_code == 200:
                # Guess extension
                content_type = r.headers.get("content-type", "").split(";")[0].strip()
                ext = mimetypes.guess_extension(content_type)
                if not ext:
                    _, path_ext = os.path.splitext(image_url.split("?")[0])
                    ext = path_ext if path_ext else ".jpg"
                    
                if ext == ".jpe" or ext == ".jpeg":
                    ext = ".jpg"
                elif not ext.startswith("."):
                    ext = "." + ext
                    
                image_name = f"{new_id}{ext}"
                dest_dir = os.path.join("public", "images")
                os.makedirs(dest_dir, exist_ok=True)
                dest_path = os.path.join(dest_dir, image_name)
                
                with open(dest_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
                print(f"Saved image to: {dest_path}")
                saved_image_path = f"images/{image_name}"
            else:
                print(f"Warning: Image download failed with status code {r.status_code}")
                saved_image_path = image_url  # Fallback to remote URL
        except Exception as e:
            print(f"Warning: Failed to download image: {e}")
            saved_image_path = image_url  # Fallback to remote URL
    else:
        print("No image URL found by scraper.")
        
    # 3. Map ingredients
    ingredients_list = scraper.ingredients()
    parsed_ingredients = []
    for ing in ingredients_list:
        parsed_ingredients.append(parse_ingredient(ing))
        
    # 4. Map instructions
    if hasattr(scraper, 'instructions_list'):
        instructions = scraper.instructions_list()
    else:
        instructions = [step.strip() for step in scraper.instructions().split('\n') if step.strip()]
        
    # 5. Map recipe fields to schema
    recipe_data = {
        "title": title,
        "source": url,
        "image": saved_image_path,
        "link": url,
        "ingredients": parsed_ingredients,
        "instructions": instructions,
        "notes": []
    }
    
    # Parse and add optional fields if available
    try:
        yields_str = scraper.yields()
        if yields_str:
            serves_match = re.search(r'\d+', yields_str)
            if serves_match:
                recipe_data["serves"] = int(serves_match.group(0))
    except Exception:
        pass
        
    try:
        prep = scraper.prep_time()
        if prep:
            recipe_data["prep_time"] = f"{prep} minutes"
    except Exception:
        pass
        
    try:
        cook = scraper.cook_time()
        if cook:
            recipe_data["cook_time"] = f"{cook} minutes"
    except Exception:
        pass
        
    # 6. Write detailed JSON file
    recipes_json_dir = os.path.join("public", "recipes-json")
    os.makedirs(recipes_json_dir, exist_ok=True)
    recipe_file_path = os.path.join(recipes_json_dir, f"{new_id}.json")
    
    with open(recipe_file_path, "w", encoding="utf-8") as f:
        json.dump(recipe_data, f, indent=2, ensure_ascii=False)
    print(f"Saved recipe details to: {recipe_file_path}")
    
    # 7. Update public/recipes.json index
    recipes.append({
        "id": new_id,
        "title": title,
        "image": saved_image_path
    })
    
    # Sort recipes by id
    recipes.sort(key=lambda x: x.get("id", 0))
    
    with open(recipes_json_path, "w", encoding="utf-8") as f:
        json.dump(recipes, f, indent=2, ensure_ascii=False)
    print(f"Updated recipe index at: {recipes_json_path}")
    print(f"Successfully imported recipe ID {new_id}!")

### Test Scraping a Recipe
Enter a URL below and run the cell to scrape and save the recipe.

In [8]:
recipe_url = "https://hintofhelen.com/sausage-baked-bean-casserole-recipe/"
scrape_recipe(recipe_url)

Scraping URL: https://hintofhelen.com/sausage-baked-bean-casserole-recipe/ ...
Error: Failed to scrape URL using recipe-scrapers: recipe-scrapers (15.11.0) exception: Website (The website 'hintofhelen.com' isn't currently supported by recipe-scrapers!
---
If you have time to help us out, please report this as a feature 
request on our bugtracker.) not supported.
